# Week 4 — Monday: Reshaping Data — Melting and Pivoting

**DATA 202 · Calvin University**

> The LORD God took the man and put him in the garden of Eden to work it and keep it. — Genesis 2:15
>
> Tending a garden means constantly reorganizing the same harvest into whatever shape is useful right now: loose vegetables sorted into baskets, baskets weighed and logged plot by plot, a season's worth of baskets summed into one total. Nothing is added and nothing is thrown away — it's the same harvest, arranged differently depending on the question being asked. That's exactly what today's two operations, **melting** and **pivoting**, do to a table.

Every question today — every prediction, every check, every task — comes back to one single idea. Get comfortable asking it out loud:

> **What is one row about?**

Once you can answer that for any table someone hands you, melting and pivoting stop being pandas syntax to memorize and become two clear, deliberate answers to two different versions of that same question.

**Today's plan (~50 min):**

| Time | Section |
|---|---|
| ~5 min | Load and inspect the data |
| ~8 min | Wide vs. Long: What Is One Row About? |
| ~19 min | Part 1 — Melting: Wide to Long (SLO 04C) |
| ~13 min | Part 2 — Pivoting and Exploding (SLO 04C) |
| ~5 min | Careful with Reshaping + what's next |

Watch for two kinds of stop-and-check along the way: **🎯 Predict First** (guess before we run the code) and **🙋 Quick Check** (a quick verbal question — no code, no pressure, just think and be ready to answer).

---
## Loading the Data

In [ ]:
import pandas as pd

DATA_PATH = "../../datasets/plot_yields.csv"
yields = pd.read_csv(DATA_PATH)
yields.head()

In [ ]:
yields.info()

**What to notice:** 18 rows, one per garden plot. Six columns — `Week1_lbs` through `Week6_lbs` — hold six weeks of harvest weight for that plot, side by side. And `Crops` holds more than one value crammed into a single cell (`"lettuce, squash"`), the same kind of multi-value column you split apart in Practice 02.

🙋 **Quick Check:** Finish this sentence out loud: *"Right now, one row of `yields` is about ___."* Now try a harder one: suppose you want the **total harvest across all 18 plots, for Week 3 only**. Could you get that with one `.groupby()` call right now, the way you did with Week 3's homelessness data? Why or why not? (Hint: what would you even group *by* — is there a column that says `"Week 3"` anywhere in this table?)

---
## Wide vs. Long: What Is One Row About? · ~8 min

Before we write a single line of `melt` or `pivot`, let's slow down and look at what those two words actually mean — with a small example, not the real 18-plot table yet.

Picture just two plots, three weeks. The exact same six harvest numbers can live in a table two completely different ways:

![A wide table with one row per plot and separate Week1/Week2/Week3 columns, next to a long table with one row per plot-per-week and a single Week column plus a single Harvest_lbs column. Arrows labeled "melt" and "pivot" connect the two, with the caption: one row = one plot's whole season (wide) vs. one row = one plot, in one week (long).](images/wide_long_diagram.png)

Look closely: **both tables contain exactly the same six numbers.** Nothing was added, nothing was thrown away. The only thing that changed is the answer to *"what is one row about?"*

* **Wide:** one row = **one plot's whole season**. To find G01's Week 2 harvest, you look up the row for `G01`, then read across to the `Week2_lbs` column.
* **Long:** one row = **one plot, in one week**. To find that same number, you look for the row where `Plot_ID` is `G01` *and* `Week` is `Week2`, then read `Harvest_lbs`.

Same fact. Two different addresses, because a row means something different in each table.

🙋 **Quick Check:** Which shape would you rather have if your next question is *"show me every week's harvest for plot G01, side by side"*? Which shape would you rather have if your next question is *"what was the average harvest per plot-week, across the whole garden"*? Notice that these aren't "right" and "wrong" tables — they're each built for a different question.

**`melt()`** turns a wide table into a long one: column *names* (like `Week1_lbs`) become values inside a new column, and the numbers they held become values inside another new column. **`pivot_table()`** does the reverse: it takes values out of a column and turns them back into column *names*. You are about to do both, on the real 18-plot dataset — and every time, before you look at the code's output, you're going to predict the answer to *what is one row about?* first.

---
## Part 1: Melting — Wide to Long (SLO 04C) · ~19 min

Right now, in the real `yields` table, one row is about **one plot's whole season**. We're going to melt the six `Week*_lbs` columns down into one `Week` column and one `Harvest_lbs` column.

🎯 **Predict First:** After we melt `yields`, what will one row be about? Finish the sentence yourself first: *"One row will be one ___, in one ___."* Then predict the row count: 18 plots × 6 weeks = how many rows?

In [ ]:
long = pd.melt(
    yields,
    id_vars=["Plot_ID", "Gardener", "Crops"],
    value_vars=["Week1_lbs", "Week2_lbs", "Week3_lbs", "Week4_lbs", "Week5_lbs", "Week6_lbs"],
    var_name="Week",
    value_name="Harvest_lbs",
)
long["Week"] = long["Week"].str.replace("_lbs", "", regex=False)
long.shape

In [ ]:
long[long['Plot_ID'] == 'G01']

Check your prediction against the output above: one row is now one **plot, in one week** — exactly the `long` shape from the diagram, just with 18 plots and 6 weeks instead of 2 and 3. 108 rows, because melting multiplies row count by however many columns you unstacked.

Look at *how* the melt call gets there: `id_vars` names the columns that get **repeated** on every new row — they still identify what plot a row belongs to. `value_vars` names the columns being **unstacked**: their *names* (`"Week1_lbs"`, `"Week2_lbs"`, ...) become the values of the new `Week` column, and their *cell contents* become the values of `Harvest_lbs`.

🙋 **Quick Check:** Now that one row means "one plot, in one week," answer the Week 3 question from the very first Quick Check: total harvest across all 18 plots, for every week. Is this a `.groupby()` you already know how to write, now that "week" lives inside the data instead of inside column names?

In [ ]:
long.groupby('Week', sort=False)['Harvest_lbs'].sum().round(1)

The season peaks in Week 3 (194.8 lbs total across the garden) and tapers off by Week 6 (67.5 lbs) — a question that was simply **unaskable** while `Week1_lbs` through `Week6_lbs` were six separate columns. That's the entire point of melting: some questions only become answerable once "which week" lives *inside* the data, not inside the column names.

---
### 🔨 Mini-Task A — Melt Just the Early Season (~3 min)

Before writing any code: if we melt *only* the first three weeks (`Week1_lbs`, `Week2_lbs`, `Week3_lbs`), what will one row of the result be about? How many rows do you expect (18 plots × how many weeks)?

Now write it. Melt those three columns into a long table called `early_long`, with the same `id_vars`, `var_name="Week"`, and `value_name="Harvest_lbs"` as above. Check the shape against your prediction.

In [ ]:
# Your code here
early_long = None


---
### 🔨 Task 1 — Find the Single Best Week (~5 min)

Using the full `long` table — where one row is one plot, in one week — find the **single highest** `Harvest_lbs` value across the *entire season*: which plot, which week, how many pounds.

1. Use `.idxmax()` on `long['Harvest_lbs']` to get the index label of the largest value.
2. Use `.loc[]` on that index to pull out the full row.
3. Assign the row's `Plot_ID` to `best_plot`, its `Week` to `best_week`, and its `Harvest_lbs` to `best_harvest`.

*(Notice this only works because one row is one plot-in-one-week: `idxmax()` on the **wide** table's `Week3_lbs` column, for example, would only ever tell you the best Week 3 — never let you compare across weeks in a single call. This is the same `idxmax()` + `.loc[]` pattern from Practice 01; what changed is what a row means, not the pandas method.)*

In [ ]:
# Your code here


---
## Part 2: Pivoting and Exploding (SLO 04C) · ~13 min

Melting answered "what if one row should mean one plot-week?" **Pivoting** asks the opposite question: what if we want one row to mean **one plot's whole season** again?

🎯 **Predict First:** If we pivot `long` back so that `Week` values become column headers again, what will one row be about? What shape (rows × columns) do you expect — and should it match the original `yields` table exactly?

In [ ]:
back_to_wide = long.pivot_table(
    index=["Plot_ID", "Gardener", "Crops"],
    columns="Week",
    values="Harvest_lbs",
).reset_index()
back_to_wide.shape

`(18, 9)` — one row is back to meaning "one plot's whole season," the exact shape we started with. `index` says which columns identify a row; `columns` says which column's *values* should become new column headers; `values` says what fills the new cells. Melting and pivoting are exact inverses of each other precisely because no information was created or destroyed in between — only the row's meaning changed, back and forth.

*(If two rows of `long` had ever shared the same `Plot_ID` **and** the same `Week` — a duplicate reading — `.pivot_table()` would need to squash them into one cell somehow, since a cell can only hold one value. Its default is to average them, via `aggfunc="mean"`. We don't have duplicates here, so you won't see it kick in, but it's the safety net this method leans on.)*

---
### 🔨 Mini-Task B — Reverse a Partial Melt (~3 min)

Before writing any code: if you pivot `early_long` (your Mini-Task A table) back to wide, what will one row mean again? What shape do you expect — 18 plots, plus how many columns?

Now write it: pivot `early_long` back to wide with `.pivot_table()` (`index=["Plot_ID", "Gardener", "Crops"]`, `columns="Week"`, `values="Harvest_lbs"`). Call the result `early_wide` and check its shape.

In [ ]:
# Your code here


### One More Reshape: What If One Row Means One Crop?

`Crops` holds several values in one cell (`"lettuce, squash"`) — the same shape problem as `Pos` in Practice 02. Right now, one row of `yields` can't answer "how many plots grow lettuce?" without you reading every cell by hand. `.explode()` fixes that by giving each crop its own row:

In [ ]:
yields_crops = yields.copy()
yields_crops["Crop_List"] = yields_crops["Crops"].str.split(", ")
exploded = yields_crops.explode("Crop_List").reset_index(drop=True)
exploded.shape

In [ ]:
exploded['Crop_List'].value_counts()

One row of `exploded` is now about **one plot, growing one crop** — so `.value_counts()` on `Crop_List` directly answers "how many plots grow this crop." Lettuce and kale tie for most widely-grown, each in 7 of the 18 plots. Notice this is a **count of plots**, not a count of pounds harvested — `.explode()` multiplied every plot's *row*, not its harvest weight, so don't read this as "how many pounds of lettuce."

---
### 🔨 Task 2 — Most Common Crop, Cross-Checked (~4 min)

1. Using `exploded['Crop_List'].value_counts()`, find which crop(s) appear in the *most* plots — check whether there's a tie for first place. Assign the highest count to `most_common_crop_count`.
2. Using `exploded['Crop_List'].nunique()`, how many *distinct* crops are grown across the whole garden this season? Assign the count to `n_distinct_crops`.

In [ ]:
# Your code here


---
## Careful with Reshaping

Every reshape you ran today was really the same move, repeated: **decide what a row should mean, then let `melt()` or `pivot_table()` get you there.** No information was added and none was thrown away — but changing what a row means changes what a careless `.sum()` or `.mean()` will silently compute:

* In the **wide** table, summing `Week3_lbs` gives you Week 3's total across every plot — straightforward, because every plot's Week 3 sits in exactly one cell.
* In the **long** table, summing the *entire* `Harvest_lbs` column gives you the whole season's total — correct, but only because every plot contributed exactly 6 rows. If one plot had *skipped* two weeks instead of harvesting zero, its 4 rows would quietly pull any "average harvest per plot-week" toward whichever plots reported more often — the exact same imbalance this week's reading found sitting inside `melt()`.

**The shape of a table is not neutral, and neither is the question "what is one row about?"** Wide format makes "compare weeks side by side" trivial and "total across all weeks" awkward. Long format flips that completely. Neither shape is *the* correct one — only the one that fits the question you're about to ask.

Here's a question neither shape can answer yet, though: **is every one of these 18 plots actually registered with the garden coordinator?** `yields` has no way to tell you — that would require a *second* table, and a new question about what connects one table's rows to another's. That's Wednesday.

---
## Coming Up

| Day | Topic | Builds on today |
|---|---|---|
| Wed | Joining tables — keys, primary/foreign keys, and the four join types | Same garden, a second table — and a new version of "what is a row about," this time asked of *two* tables at once |
| Week 5 | Clustering & Dimensionality Reduction | Finding groups the data suggests, instead of ones we choose (like `Plot_ID` or `Week`) in advance |